# ema-second-moment — worked example 3: From v-Buffer to Adaptive Scale: Seeing the Denominator

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `ema-second-moment`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Adam divides the gradient by `sqrt(v) + eps` to form the parameter update. A large `v` (parameter received consistently large gradients) produces a small effective learning rate — this is the adaptive part. A small `v` (rarely or weakly-updated parameter) produces a large effective learning rate. The second-moment buffer is the sole source of Adam's per-coordinate adaptivity.

## Worked solution

We compute the step scales for three parameters that have accumulated different gradient histories, then verify the inverse relationship.

**Step 1 — fill the v buffers:** Run 50 steps for each parameter with different gradient magnitudes: 10.0, 1.0, and 0.1. After 50 steps, `v` will be close to `g^2` for each.

**Step 2 — compute step scales:** `scale = 1 / (v.sqrt() + eps)`. A parameter with `v ≈ 100` (g=10) gets scale ≈ `1/(10 + eps) ≈ 0.1`. A parameter with `v ≈ 0.01` (g=0.1) gets scale ≈ `1/(0.1 + eps) ≈ 10`. The scales span two orders of magnitude automatically.

**Step 3 — verify ordering:** We confirm that param with the largest gradient history has the smallest step scale, and vice versa. This is the core mechanism that makes Adam robust to ill-conditioned loss surfaces.

In [ ]:
import torch as t

t.manual_seed(7)

beta2 = 0.999
eps = 1e-8

# Three parameters with gradient magnitudes: big, medium, small
grads = t.tensor([10.0, 1.0, 0.1])
v = t.zeros(3)

for _ in range(50):
    v.copy_(beta2 * v + (1 - beta2) * grads.pow(2))

print("v after 50 steps:", [f"{x:.4f}" for x in v.tolist()])
print("g^2 (target):     ", [f"{x:.4f}" for x in grads.pow(2).tolist()])

step_scale = 1.0 / (v.sqrt() + eps)
print("Step scales:      ", [f"{x:.4f}" for x in step_scale.tolist()])

# Parameter with largest gradient history => smallest step scale
assert step_scale.argmin().item() == 0, "Param 0 (g=10) should have the smallest step scale"
assert step_scale.argmax().item() == 2, "Param 2 (g=0.1) should have the largest step scale"
print("Verified: large-gradient param gets small step scale, small-gradient param gets large step scale.")